# CDDFuse-AG Paper-Faithful MIF Training

120 ep / 2-phase / **batch 16** / **α₃=10** / AMP fp16 — đúng paper text §5.1.

Inputs: dataset Harvard medical đã split train/test sẵn (`MyDatasets/{modal}/train|test/`):
- Train: 160 CT + 245 PET + 333 SPECT = **738 cặp**
- Test: 24 mỗi modality = **72 cặp**

Train CDDFuse-AG = Adaptive Gating + Saliency-guided Pixel.

## Cell 1 — Config

In [ ]:
VARIANT     = 'Combined-Gated-Saliency'   # CDDFuse-AG = Adaptive Gating + Saliency-guided Pixel
REPO_URL    = 'https://github.com/kienvbhp872004/Image-Fusion.git'
REPO_BRANCH = 'main'
EPOCHS      = 120
EPOCH_GAP   = 40
BATCH       = 16                          # paper text §5.1 (was 8 trong code release)
SEED        = 42
USE_AMP     = True

import os, subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
GPU_NAME = r.stdout.strip().splitlines()[0] if r.stdout else 'unknown'
print(f'[gpu] {GPU_NAME}')

## Cell 2 — Clone repo, install deps, downgrade torch cho P100 sm_60

In [ ]:
!git clone --branch $REPO_BRANCH $REPO_URL /kaggle/working/Image-Fusion
%cd /kaggle/working/Image-Fusion
!pip install -q einops==0.4.1 kornia==0.6.12 h5py tqdm scikit-image

# Downgrade torch để support P100 sm_60 (torch 2.6+ drop Pascal). 2.5.1 là bản cuối.
!pip uninstall -y torch torchvision torchaudio 2>&1 | tail -3
!pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!python -c "import torch; print('[torch]', torch.__version__, 'cuda:', torch.cuda.is_available(), 'cap:', torch.cuda.get_device_capability(0) if torch.cuda.is_available() else 'N/A')"

## Cell 3 — Stage Harvard medical với split train/test có sẵn (MyDatasets/)

In [ ]:
import shutil, pathlib, glob

def find_dataset_root(slug):
    for c in [f'/kaggle/input/{slug}', f'/kaggle/input/datasets/kienvbhp1234/{slug}']:
        if pathlib.Path(c).exists(): return pathlib.Path(c)
    m = glob.glob(f'/kaggle/input/**/{slug}', recursive=True)
    if m: return pathlib.Path(m[0])
    raise FileNotFoundError(slug)

TRAIN_SRC = find_dataset_root('harvard-medical-train')   # 738 cặp train (160+245+333)
TEST_SRC  = find_dataset_root('harvard-medical-fusion')  # 72 cặp test (24×3)
print(f'[paths] train={TRAIN_SRC}\n[paths] test ={TEST_SRC}')

# Stage thành cấu trúc MyDatasets/{modal}/{train,test}/{src,MRI}/
POOL = pathlib.Path('/kaggle/working/Image-Fusion/Havard-Medical-Image-Fusion-Datasets-main/Havard-Medical-Image-Fusion-Datasets-main/MyDatasets')
POOL.mkdir(parents=True, exist_ok=True)

for modal in ['CT-MRI', 'PET-MRI', 'SPECT-MRI']:
    sub = modal.split('-')[0]
    # Train: copy từ harvard-medical-train
    train_src = TRAIN_SRC / modal
    if train_src.exists():
        dst = POOL / modal / 'train'
        dst.mkdir(parents=True, exist_ok=True)
        shutil.copytree(train_src, dst, dirs_exist_ok=True)
    # Test: copy từ harvard-medical-fusion
    test_src = TEST_SRC / modal
    if test_src.exists():
        dst = POOL / modal / 'test'
        dst.mkdir(parents=True, exist_ok=True)
        shutil.copytree(test_src, dst, dirs_exist_ok=True)
    n_train = len(list((POOL / modal / 'train' / sub).glob('*.png')))
    n_test  = len(list((POOL / modal / 'test'  / sub).glob('*.png')))
    print(f'[stage] {modal}: train={n_train} test={n_test}')

## Cell 4 — Pre-process: extract patches → h5

In [ ]:
%cd /kaggle/working/Image-Fusion/models/MMIF-CDDFuse
!python dataprocessing_MIF.py

## Cell 5 — Train CDDFuse-AG (120 ep, 2-phase, batch 16, α₃=10, AMP fp16)

In [ ]:
amp_flag = '--amp' if USE_AMP else ''
# Hyperparam paper text §5.1: batch 16, α1=1, α2=2, α3=10, α4=2
!python train_MIF.py \
    --variant     $VARIANT \
    --num_epochs  $EPOCHS \
    --epoch_gap   $EPOCH_GAP \
    --batch       $BATCH \
    --coeff_tv    10.0 \
    --coeff_decomp 2.0 \
    --seed        $SEED \
    --output      /kaggle/working/ \
    $amp_flag

## Cell 6 — Inference + per-image metrics

In [ ]:
import glob
ckpts = sorted(glob.glob(f'/kaggle/working/CDDFuse-{VARIANT}_MIF_*.pth'))
assert ckpts, 'No checkpoint found from Cell 5'
CKPT = ckpts[-1]
OUT_DIR = f'/kaggle/working/CDDFuse-{VARIANT}-PaperMIF'
print(f'[ckpt] {CKPT}')

# Stage test 72 cặp (24 × 3 modal) sang cấu trúc data/reference/ cho evaluate_cddfuse.py
import shutil, pathlib
MYDS = pathlib.Path('/kaggle/working/Image-Fusion/Havard-Medical-Image-Fusion-Datasets-main/Havard-Medical-Image-Fusion-Datasets-main/MyDatasets')
TEST_OUT = pathlib.Path('/kaggle/working/Image-Fusion/data/reference')
for modal in ['CT-MRI', 'PET-MRI', 'SPECT-MRI']:
    sub = modal.split('-')[0]
    src_test = MYDS / modal / 'test'
    if not src_test.exists():
        print(f'[skip] {modal} test missing'); continue
    (TEST_OUT / modal).mkdir(parents=True, exist_ok=True)
    shutil.copytree(src_test, TEST_OUT / modal, dirs_exist_ok=True)
    n = len(list((TEST_OUT / modal / sub).glob('*.png')))
    print(f'[test ] {modal}: {n} cặp staged')

variant_flag = f'--variant {VARIANT}' if VARIANT != 'CDDFuse' else ''
for modal in ['CT', 'PET', 'SPECT']:
    !python evaluate_cddfuse.py \
        $variant_flag \
        --modal         $modal \
        --ckpt          $CKPT \
        --harvard_root  /kaggle/working/Image-Fusion/data/reference \
        --out_dir       $OUT_DIR \
        --save_perimage

## Cell 7 — Package output

In [ ]:
import tarfile, json, hashlib, datetime, os, glob
h = hashlib.sha256()
with open(CKPT, 'rb') as f:
    for chunk in iter(lambda: f.read(8192), b''): h.update(chunk)
stamp = {
    'variant_name': f'CDDFuse-{VARIANT}-PaperMIF',
    'based_on':     'CDDFuse paper text §5.1 (batch 16, α3=10)',
    'datetime':     datetime.datetime.utcnow().isoformat() + 'Z',
    'train_mode':   'full_paper_120ep_2phase',
    'gpu':          GPU_NAME,
    'epochs':       EPOCHS,
    'epoch_gap':    EPOCH_GAP,
    'batch':        BATCH,
    'amp':          USE_AMP,
    'seed':         SEED,
    'coeff_tv':     10.0,
    'coeff_decomp': 2.0,
    'ckpt_sha256':  h.hexdigest(),
}
with open(f'{OUT_DIR}/_ablation_stamp.json', 'w') as f:
    json.dump(stamp, f, indent=2)

tar_path = f'/kaggle/working/CDDFuse-{VARIANT}-PaperMIF_results.tar.gz'
with tarfile.open(tar_path, 'w:gz') as tar:
    tar.add(OUT_DIR, arcname=f'CDDFuse-{VARIANT}-PaperMIF')
    tar.add(CKPT, arcname=f'CDDFuse-{VARIANT}-PaperMIF.pth')
    hist_files = glob.glob(f'/kaggle/working/CDDFuse-{VARIANT}_MIF_*_train_history.json')
    for hf in hist_files:
        tar.add(hf, arcname=os.path.basename(hf))
    split_json = '/kaggle/working/Image-Fusion/models/MMIF-CDDFuse/data/MIF_split.json'
    if os.path.exists(split_json):
        tar.add(split_json, arcname='MIF_split.json')
print('[done]', tar_path)